# 03 — Revision Experiments (Edge-IIoTset)
## Response to IEEE Access reviewers — Access-2026-34466

Fatma Mohammed Dhaou · University of Tabuk · fdhaou@ut.edu.sa

Each module is labelled with the reviewer comment(s) it answers. Run top to bottom. Copy each module's printed output back for integration into the manuscript and the response-to-reviewers document.

| Module | Answers |
|---|---|
| 1 Setup & clean | — |
| 2 Training-only leakage screen | **R2.2** |
| 3 Alternative screens: MI, RFE, adversarial validation | **R1.4, R1.5, R2.3** |
| 4 Threshold sensitivity (0.80/0.85/0.90) | **R1.15** |
| 5 Ablation cascade | **R1.2, R1.7** |
| 6 Near-duplicate screen | **R1.7** |
| 7 Repeated stratified CV + significance | **R1.11** |
| 8 1D-CNN (confirmatory deep model) | **R1.10** |
| 9 Extended metrics (MCC, balanced acc, FPR/FNR) + Fig 2 correction | **R1.13, R2.1** |
| 10 Regenerated large-font figures | **R1.16** |


## Module 1 — Setup, load, clean

Same drop-then-deduplicate ordering as the main analysis. Asserts 152,245 rows.


In [ ]:
# Colab: uncomment if needed
# from google.colab import files; up=files.upload(); CSV_PATH=list(up.keys())[0]
CSV_PATH = 'ML-EdgeIIoT-dataset.csv'

import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, balanced_accuracy_score,
                             confusion_matrix)
from scipy import stats

RS = 42
np.random.seed(RS)

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]:,} x {df.shape[1]}')

DROP_13 = ['frame.time','ip.src_host','ip.dst_host','arp.dst.proto_ipv4','arp.src.proto_ipv4',
           'http.file_data','http.request.uri.query','http.request.full_uri','http.referer',
           'tcp.options','tcp.payload','tcp.srcport','mqtt.msg']
df = df.drop(columns=[c for c in DROP_13 if c in df.columns])
df = df.dropna().drop_duplicates().reset_index(drop=True)
assert len(df)==152245, f'Expected 152,245, got {len(df):,}'
print(f'After clean: {len(df):,} rows  -> CHECKPOINT PASSED')

CATEG = ['http.request.method','http.request.version','dns.qry.name.len',
         'mqtt.conack.flags','mqtt.protoname','mqtt.topic']
df_enc = df.copy()
for c in CATEG:
    if c in df_enc: df_enc[c] = df_enc[c].astype('category').cat.codes

y = df_enc['Attack_label'].astype(int)
ymul = df_enc['Attack_type']
X = df_enc.drop(columns=['Attack_label','Attack_type'])
print(f'Features: {X.shape[1]}  (expect 48)')
assert X.shape[1]==48


## Module 2 — Training-only leakage screen  (R2.2)

Reviewer 2 correctly notes the screen was computed on the full cleaned dataset. Here the correlation is computed on the **training partition only**, and the flagged features are then removed from both partitions. This is the split used by every experiment below.


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=RS, stratify=ymul)
ymul_tr = ymul.loc[Xtr.index]; ymul_te = ymul.loc[Xte.index]

corr_train = Xtr.corrwith(ytr).abs().sort_values(ascending=False)
print('Top 8 |Pearson r| with Attack_label (TRAIN ONLY):')
print(corr_train.head(8).to_string(float_format=lambda v:f'{v:.4f}'))

LEAK = corr_train[corr_train > 0.85].index.tolist()
print(f'\nFlagged at |r|>0.85 (train only): {LEAK}')
print('Manuscript (full-data) flagged: dns.qry.name.len, mqtt.topic, mqtt.protoname, mqtt.conack.flags')
print('-> Confirm these are the SAME four.')

COLS48 = list(X.columns)
COLS44 = [c for c in COLS48 if c not in LEAK]
print(f'\nUncontrolled: {len(COLS48)}   Leakage-controlled: {len(COLS44)}')


## Module 3 — Alternative screens: MI, RFE, adversarial validation  (R1.4, R1.5, R2.3)

Three complementary detectors, all fitted on the training partition, compared against the Pearson screen:

- **Mutual information** — captures *non-linear* univariate dependence (R2.3, R1.5). Computed on the raw (unscaled) values.
- **RFE** with a Random Forest estimator — *multivariate* ranking (R1.5).
- **Adversarial validation** — trains a classifier to distinguish the train partition from the test partition. AUC ≈ 0.5 means the split itself is clean, i.e. the perfect accuracy is **not** train/test split contamination but an intrinsic feature–label shortcut. This distinguishes two kinds of leakage.

The key question for R1.4/R1.5: do MI and RFE flag features that the Pearson screen *misses*? If so, that is direct evidence that correlation screening alone is inadequate — the paper's thesis.


In [ ]:
# --- Mutual information (train, raw values) ---
mi = pd.Series(mutual_info_classif(Xtr, ytr, discrete_features='auto', random_state=RS),
               index=Xtr.columns).sort_values(ascending=False)
print('Top 10 by Mutual Information (train):')
print(mi.head(10).to_string(float_format=lambda v:f'{v:.4f}'))

pearson_top4 = set(corr_train.head(4).index)
mi_top10 = set(mi.head(10).index)
print(f'\nPearson top-4: {sorted(pearson_top4)}')
print(f'Features in MI top-10 but NOT Pearson top-4: {sorted(mi_top10 - pearson_top4)}')
print('  -> any such feature is a shortcut the linear screen would miss.')


In [ ]:
# --- RFE with Random Forest (multivariate ranking) ---
# Slower: retrains a small forest as it eliminates. A few minutes is normal.
rfe = RFE(RandomForestClassifier(n_estimators=50, max_depth=15, random_state=RS, n_jobs=-1),
          n_features_to_select=10, step=2)
rfe.fit(Xtr, ytr)
rfe_top = [c for c,keep in zip(Xtr.columns, rfe.support_) if keep]
print('RFE-selected top 10 features:')
for c in rfe_top: print('  ', c)
print(f'\nRFE-selected but NOT in Pearson top-4: {sorted(set(rfe_top)-pearson_top4)}')


In [ ]:
# --- Adversarial validation: is the train/test split itself leaky? ---
adv_X = pd.concat([Xtr, Xte]); adv_y = np.r_[np.zeros(len(Xtr)), np.ones(len(Xte))]
aXtr, aXte, aytr, ayte = train_test_split(adv_X, adv_y, test_size=0.3, random_state=RS, stratify=adv_y)
adv = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RS, n_jobs=-1).fit(aXtr, aytr)
adv_auc = roc_auc_score(ayte, adv.predict_proba(aXte)[:,1])
print(f'Adversarial-validation AUC (train vs test): {adv_auc:.4f}')
print('~0.5 => the split is clean; the perfect accuracy is an intrinsic feature-label')
print('shortcut, NOT train/test contamination. (A value >>0.5 would indicate split leakage.)')


## Module 4 — Threshold sensitivity  (R1.15)

The 0.85 cutoff is a qualitative judgement, not a formal criterion. This shows the flagged set is stable across nearby thresholds.


In [ ]:
for t in [0.80, 0.85, 0.90]:
    flagged = corr_train[corr_train>t].index.tolist()
    print(f'|r| > {t:.2f}  ->  {len(flagged)} features: {flagged}')
print('\nStable flagging across thresholds supports that the choice is not arbitrary.')


## Module 5 — Ablation cascade  (R1.2, R1.7)

**The central experiment for R1.2.** Starting from the leakage-controlled (44-feature) set, iteratively remove the most important Random Forest features and re-evaluate. If accuracy stays near 1.000 as many features are stripped, the shortcut is **redundantly encoded** across the feature space — which is exactly the paper's claim, shown rather than inferred.


In [ ]:
rf_full = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1)
rf_full.fit(Xtr[COLS44], ytr)
order = pd.Series(rf_full.feature_importances_, index=COLS44).sort_values(ascending=False).index.tolist()

ks = [0,2,4,8,12,16,20,24,28,32,36,40]
abl = []
for k in ks:
    keep = order[k:]                       # remove the top-k most important
    if len(keep) < 2: break
    m = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1)
    m.fit(Xtr[keep], ytr)
    acc = accuracy_score(yte, m.predict(Xte[keep]))
    abl.append({'top_k_removed':k, 'features_left':len(keep), 'accuracy':acc})
    print(f'removed top {k:>2}  |  {len(keep):>2} left  |  acc = {acc:.4f}')
abl = pd.DataFrame(abl)
print('\nIf accuracy holds near 1.000 well down the list => redundant shortcut encoding.')


## Module 6 — Near-duplicate screen  (R1.7)

Only exact duplicates were removed in the main analysis. Near-duplicates spanning the train/test split are a classic cause of inflated accuracy. Here we quantify how many test records have a near-identical neighbour in the training set, using a hash of the discretised feature vector on the controlled feature set.


In [ ]:
# Discretise continuous features into coarse bins, hash the row, count cross-split collisions
def sig(frame, cols, bins=50):
    z = frame[cols].copy()
    for c in cols:
        if z[c].nunique() > bins:
            z[c] = pd.qcut(z[c].rank(method='first'), q=bins, labels=False, duplicates='drop')
    return pd.util.hash_pandas_object(z, index=False)

htr = set(sig(Xtr, COLS44))
hte = sig(Xte, COLS44)
near = hte.isin(htr).mean()
print(f'Share of TEST records with a near-identical TRAIN neighbour (coarse 50-bin hash): {near*100:.2f}%')
print('Reported as a quantified caveat; high values would indicate near-duplicate contamination.')


## Module 7 — Repeated stratified CV + significance  (R1.11)

Replaces the single split with 5-fold × 3-repeat stratified CV. Reports mean ± std accuracy per model per feature set, and a Wilcoxon signed-rank test on the per-fold accuracy difference (uncontrolled − controlled) for the linear model vs Random Forest.

*Runtime note:* Gradient Boosting is the slow learner; if this is too slow on Colab free tier, reduce to `n_repeats=1` or drop GB from the loop.


In [ ]:
models = {
  'RandomForest': lambda: RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1),
  'GradBoost':    lambda: GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=RS),
  'LogReg':       lambda: LogisticRegression(penalty='l2', max_iter=2000, random_state=RS),
  'MLP':          lambda: MLPClassifier(hidden_layer_sizes=(64,32), early_stopping=True, random_state=RS),
}
NEEDS_SCALE = {'LogReg','MLP'}
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RS)

import collections
fold_acc = collections.defaultdict(list)   # (model, featset) -> [acc per fold]
for name, cols in [('48', COLS48), ('44', COLS44)]:
    Xc = X[cols].values; yv = y.values; strat = ymul.values
    for tr, te in rskf.split(Xc, strat):
        sc = StandardScaler().fit(Xc[tr])
        Xtr_s, Xte_s = sc.transform(Xc[tr]), sc.transform(Xc[te])
        for mname, mk in models.items():
            a,b = (Xtr_s,Xte_s) if mname in NEEDS_SCALE else (Xc[tr],Xc[te])
            m = mk().fit(a, yv[tr])
            fold_acc[(mname,name)].append(accuracy_score(yv[te], m.predict(b)))
    print(f'feature set {name}: done')

print('\nMean +/- std accuracy over 15 folds:')
for mname in models:
    for fs in ['48','44']:
        arr = np.array(fold_acc[(mname,fs)])
        print(f'  {mname:<12} [{fs}]  {arr.mean():.4f} +/- {arr.std():.4f}')

# Wilcoxon: does removing the 4 fields change accuracy? (per model)
print('\nWilcoxon signed-rank (48 vs 44), per model:')
for mname in models:
    a = np.array(fold_acc[(mname,'48')]); b = np.array(fold_acc[(mname,'44')])
    try:
        w,p = stats.wilcoxon(a,b)
        print(f'  {mname:<12} median delta={np.median(a-b):+.4f}  p={p:.3g}')
    except Exception as e:
        print(f'  {mname:<12} (identical folds -> no variance)  median delta={np.median(a-b):+.4f}')


## Module 8 — 1D-CNN, confirmatory deep model  (R1.10)

A compact 1-D convolutional network, the architecture most common in the IoT-IDS literature for tabular flow features. Evaluated on both feature sets. If it too retains near-perfect accuracy after leakage control, that is *additional* confirmation that the shortcut is recoverable by yet another non-linear architecture.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models as km
tf.random.set_seed(RS)

def run_cnn(cols):
    sc = StandardScaler().fit(Xtr[cols])
    A = sc.transform(Xtr[cols])[...,None]; B = sc.transform(Xte[cols])[...,None]
    net = km.Sequential([
        layers.Input(shape=(len(cols),1)),
        layers.Conv1D(32,3,activation='relu',padding='same'),
        layers.Conv1D(64,3,activation='relu',padding='same'),
        layers.GlobalMaxPooling1D(),
        layers.Dense(64,activation='relu'), layers.Dropout(0.3),
        layers.Dense(1,activation='sigmoid')])
    net.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    es = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    net.fit(A, ytr.values, validation_split=0.1, epochs=20, batch_size=512, callbacks=[es], verbose=0)
    prob = net.predict(B, verbose=0).ravel(); pred = (prob>0.5).astype(int)
    return accuracy_score(yte,pred), f1_score(yte,pred), roc_auc_score(yte,prob)

for name, cols in [('Uncontrolled (48)',COLS48),('Leakage-controlled (44)',COLS44)]:
    acc,f1,auc = run_cnn(cols)
    print(f'1D-CNN | {name:<24} acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')


## Module 9 — Extended metrics + Figure 2 correction  (R1.13, R2.1)

Adds MCC, balanced accuracy, FPR and FNR to the binary results, and prints the exact Random Forest confusion-matrix counts so the manuscript text matches the figure (Reviewer 2: the RF binary model makes **one** error, a single false positive — not zero).


In [ ]:
def full_metrics(cols):
    rows=[]
    sc = StandardScaler().fit(Xtr[cols])
    for mname, mk in models.items():
        a,b = (sc.transform(Xtr[cols]),sc.transform(Xte[cols])) if mname in NEEDS_SCALE else (Xtr[cols],Xte[cols])
        m = mk().fit(a,ytr); pred=m.predict(b)
        tn,fp,fn,tp = confusion_matrix(yte,pred).ravel()
        rows.append({'Model':mname,'Acc':accuracy_score(yte,pred),
                     'MCC':matthews_corrcoef(yte,pred),'BalAcc':balanced_accuracy_score(yte,pred),
                     'FPR':fp/(fp+tn),'FNR':fn/(fn+tp),'FP':fp,'FN':fn})
    return pd.DataFrame(rows)

print('Leakage-controlled (44) — extended metrics:')
print(full_metrics(COLS44).to_string(index=False, float_format=lambda v:f'{v:.4f}'))

rf = RandomForestClassifier(n_estimators=200,max_depth=20,random_state=RS,n_jobs=-1).fit(Xtr[COLS44],ytr)
tn,fp,fn,tp = confusion_matrix(yte, rf.predict(Xte[COLS44])).ravel()
print(f'\nRF confusion counts (controlled): TN={tn} FP={fp} FN={fn} TP={tp}')
print(f'RF total errors: {fp+fn} of {len(yte):,}  (Reviewer 2: confirm this is 1, not 0)')


## Module 10 — Regenerated large-font figures  (R1.16)

Re-exports the ROC, confusion, importance and SHAP figures at 300 dpi with enlarged fonts. Download the PNGs and replace the current figures in the manuscript.


In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import roc_curve
plt.rcParams.update({'font.size':14,'axes.titlesize':15,'axes.labelsize':14,
                     'xtick.labelsize':12,'ytick.labelsize':12,'legend.fontsize':11})

sc = StandardScaler().fit(Xtr[COLS44])
fitted={}
for mname,mk in models.items():
    a,b=(sc.transform(Xtr[COLS44]),sc.transform(Xte[COLS44])) if mname in NEEDS_SCALE else (Xtr[COLS44],Xte[COLS44])
    m=mk().fit(a,ytr); fitted[mname]=(m, m.predict_proba(b)[:,1], m.predict(b))

plt.figure(figsize=(8,6))
for mname,(m,prob,pred) in fitted.items():
    fpr,tpr,_=roc_curve(yte,prob)
    plt.plot(fpr,tpr,lw=2,label=f'{mname} (AUC={roc_auc_score(yte,prob):.4f})')
plt.plot([0,1],[0,1],'k--',lw=1,alpha=.5)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Binary Detection (Leakage-Controlled)')
plt.legend(loc='lower right'); plt.tight_layout(); plt.savefig('fig1_roc_v2.png',dpi=300); plt.show()

cm=confusion_matrix(yte, fitted['RandomForest'][2])
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,annot_kws={'size':16},
            xticklabels=['Normal','Attack'],yticklabels=['Normal','Attack'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Random Forest — Binary, Leakage-Controlled')
plt.tight_layout(); plt.savefig('fig2_confusion_v2.png',dpi=300); plt.show()
print('Saved fig1_roc_v2.png, fig2_confusion_v2.png (download and replace in manuscript).')


---

**Done.** Copy each module's output back. Modules 2–9 feed specific manuscript edits and response-to-reviewers entries; Module 10 produces replacement figures. The CICIoT2023 generalisation is in the separate notebook `04_cross_dataset_ciciot2023.ipynb`.
